# Lorenz-63 CSE reproduction: full-state assimilation and xyz control


## Imports


In [ ]:
import importlib
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from class_da_sys import da_system
from class_Lorenz63 import run_RK4
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)
from class_cse_qiwen import (
    build_and_apply_perturbation,
    detect_regime_changes,
    find_past_safe_trajectory,
    integrate_and_align,
    lorenz63,
    plot_found_past_safe_trajectory,
    plot_cse_cycle_diagnostics,
    plot_cse_initialization,
    plot_failed_event_diagnostic,
    plot_3d_nature_run_comparison,
    plot_filter_diagnostic_during_control,
    plot_lorenz_attractor_aligned_cse_starts,
    plot_past_safe_search_visual_diagnostic,
    plot_regime_crossings,
    run_past_safe_trajectory_test,
    run_single_cse as run_single_cse_core,
    run_single_cse_mini_test,
    summarize_cse_batch_results,
    summarize_cse_batch_results_by_D,
    run_perturbation_unit_test,
)


## DA System Setup


In [ ]:
BASE_PATH = Path.cwd()
PathFigures = BASE_PATH / 'CSE_Qiwen_xyz_control_nature_qiwenevents'
PathFigures.mkdir(parents=True, exist_ok=True)

# This notebook assimilates x, y, z as in run_cse_qiwen.ipynb,
# and applies CSE perturbations to x, y, and z.
# control_indices=None makes build_and_apply_perturbation skip component masking
# and perturb x, y, and z, which is the original CSE case in Figure 3(a) of the paper.
CONTROL_INDICES = None
CONTROL_LABEL = 'xyz'
REGIME_REFERENCE_MODE = 'nature'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(BASE_PATH / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04


## Load OSSE Histories


In [ ]:
with open(BASE_PATH / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(BASE_PATH / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
regime_change_info = detect_regime_changes(
    x_nature,
    BASE_PATH,
    position=position,
    output_filename='cse_qiwen_xyz_control_change_region_time_qiwen_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

change_region_time = regime_change_info['change_region_time']
check_position = regime_change_info['check_position']
t_start_series = regime_change_info['t_start_series']
t_start_residence = regime_change_info['t_start_residence']
mean_crossing_interval = regime_change_info['mean_crossing_interval']


In [ ]:
# Independent reference reconstruction from cell 2 of 06_cse_batch_z_only.ipynb.
selected = []
for i, crossing in enumerate(change_region_time):
    if crossing >= 500 and crossing - change_region_time[i - 1] > 385:
        selected.append(crossing)
referencia = selected[:100]

print('t_start_series[:40]:', t_start_series[:40])
print('referencia[:40]:', referencia[:40])
assert t_start_series[:40] == referencia[:40]

diego_selection = globals().get('diego_t_start_series', globals().get('t_start_series_diego'))
if diego_selection is None:
    previous_notebook = BASE_PATH / 'run_cse_qiwen_z_control_member0.ipynb'
    if previous_notebook.exists():
        diego_selection = []
        for idx, crossing in enumerate(change_region_time):
            if crossing < 500 or not x_nature[0, crossing - 1] > 0 or idx == 0:
                continue
            if crossing - change_region_time[idx - 1] > 385:
                diego_selection.append(crossing)
        print('Notebook anterior disponible; selección diego reconstruida para la comparación.')
if diego_selection is not None:
    n_different = sum(a != b for a, b in zip(t_start_series, diego_selection)) + abs(len(t_start_series) - len(diego_selection))
    print('Número de eventos que difieren respecto a diego:', n_different)
else:
    print('Selección diego del notebook anterior no disponible.')


## Visual Check of Regime Crossings


In [ ]:
fig_regime_crossings, axes_regime_crossings, regime_crossings_path = plot_regime_crossings(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
    output_filename='regime_crossings_qiwen_qiwenevents.png',
    event_selection=EVENT_SELECTION,
)


## Lorenz Attractor View of Regime-Change Points


In [ ]:
fig_lorenz_starts, axes_lorenz_starts, lorenz_starts_path = plot_lorenz_attractor_aligned_cse_starts(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
)
lorenz_starts_tagged_path = lorenz_starts_path.with_name(f'{lorenz_starts_path.stem}{OUTPUT_TAG}{lorenz_starts_path.suffix}')
lorenz_starts_path.replace(lorenz_starts_tagged_path)
lorenz_starts_path = lorenz_starts_tagged_path


## CSE Parameters


In [ ]:
D = 0.05
amp = 1.0 / D
experiment_step = 1000
EVENT_INDEX = 1

solver_params = {
    'model_func': lorenz63,
    'sigma': 10.0,
    'beta': 8.0 / 3.0,
    'rho': 28.0,
    'dt': fcst_dt,
}

print('position:', position)


## Initialize CSE Experiment


In [ ]:
t_start = t_start_series[EVENT_INDEX]
t0 = (t_start - position) - (t_start - position) % Ta
osse_start_index = (t0 - osse_summary['t0']) // Ta
if osse_summary['t0'] + osse_start_index * Ta != t0:
    raise ValueError('t0 is not aligned with the OSSE grid')

Xa_initial = np.matrix(osse_Xa_history[osse_start_index])  # (xdim, edim)
x_original = np.asarray(x_nature[:, t0]).ravel().copy()

Xa_history_control = np.zeros((experiment_step + 1, das.xdim, das.edim))
Xa_history_control[0] = osse_Xa_history[osse_start_index]
evo_history = np.zeros((3, experiment_step * Ta + 1))
evo_history[:, 0] = x_original

assert Xa_initial.shape == (das.xdim, das.edim)
rmse_init = np.linalg.norm(np.asarray(Xa_initial).mean(axis=1) - x_original) / np.sqrt(3)

print('rmse_init:', rmse_init)
print('t0:', t0)
print('osse_start_index:', osse_start_index)
print('np.sign(x_original[0]):', np.sign(x_original[0]))


## Visual Check of Initialization


In [ ]:
# Pre-run visual check of the CSE initialization.
# This confirms that t0 is correctly placed before the selected event and
# shows where the inherited OSSE analysis ensemble sits relative to nature.
fig_cse_init, ax_cse_init, cse_init_path = plot_cse_initialization(
    x_nature=x_nature,
    Xa_initial=Xa_initial,
    t0=t0,
    t_start=t_start,
    osse_start_index=osse_start_index,
    rmse_init=rmse_init,
    path_figures=PathFigures,
    output_filename='cse_initialization_qiwen_qiwenevents.png',
)


## Build and Apply Perturbation


In [ ]:
# build_and_apply_perturbation is imported from class_cse_qiwen.py.
# The implementation is kept in the module so the CSE loop below stays focused on experiment logic.


## Perturbation Unit Test


In [ ]:
# Unit test for the perturbation builder.
# It checks that each perturbation has norm D, that the stored history starts
# at the original state and has length Ta, and that the controlled final state
# differs from the free integration over the same DA cycle.
perturbation_unit_test = run_perturbation_unit_test(
    x_nature=x_nature,
    t0=t0,
    edim=das.edim,
    Xa_initial=Xa_initial,
    position=position,
    D=D,
    Ta=Ta,
    solver_params=solver_params,
    cycle=0,
    control_indices=CONTROL_INDICES,
)


## Find a Past Safe Trajectory


In [ ]:
# integrate_and_align and find_past_safe_trajectory are imported from class_cse_qiwen.py.
# Keeping the safe-search implementation in the module makes the control loop easier to read.


## Controlled CSE Loop


In [ ]:
# plot_found_past_safe_trajectory is imported from class_cse_qiwen.py.
# The controlled loop below keeps only the experiment logic.

N_TEST = experiment_step
SAVE_CYCLE_FIGS = False
SAVE_PAST_SAFE_FIGS = SAVE_CYCLE_FIGS
rng = np.random.default_rng(1)
obs_error_std = np.sqrt(2.0)
# Reset controlled experiment state so rerunning this cell is reproducible.
x_original = np.asarray(x_nature[:, t0]).ravel().copy()
Xa = Xa_initial.copy()
Xa_history_control = np.zeros((experiment_step + 1, das.xdim, das.edim))
Xa_history_control[0] = osse_Xa_history[osse_start_index]
evo_history = np.zeros((3, experiment_step * Ta + 1))
evo_history[:, 0] = x_original

analysis_rmse = []
counterfactual_divergence = []
p_change_history = []
cycle_t_now_history = []
Xa_cycle_initial_history = []
xf_4d_box_history = []
unchange_ens_history = []
change_ens_history = []
yo_history = []
perturbed_time = []
false_alarm_perturbed_time = []
success_perturbed_time = []
false_alarm_cycles = []
success_candidate_cycles = []
branch_history = []
branch_counts = {'free': 0, 'current_safe_control': 0, 'past_safe_control': 0}
past_safe_metadata = []
safe_search_log = []
perturbation_norm_history = []
cant_control = False
cant_control_cycle = None
evol = None
hidden_forecast_needs_recompute = True
initial_regime_value = evo_history[0, 0]

for time in range(N_TEST):
    if not np.all(evo_history[0, 0:time * Ta] * initial_regime_value > 0):
        cant_control = True
        cant_control_cycle = time
        break

    if hidden_forecast_needs_recompute:
        evol = run_RK4(
            lorenz63,
            10.0,
            8.0 / 3.0,
            28.0,
            x_original,
            fcst_dt,
            fcst_dt * position,
        ).integrate()[:position + 1, :].T
    else:
        evol_extension = run_RK4(
            lorenz63,
            10.0,
            8.0 / 3.0,
            28.0,
            np.asarray(evol[:, -1]).ravel(),
            fcst_dt,
            fcst_dt * Ta,
        ).integrate()
        evol = np.concatenate([evol[:, Ta:], evol_extension[1:Ta + 1, :].T], axis=1)

    t_now = t0 + time * Ta
    cycle_perturbed = False
    cycle_t_now_history.append(t_now)
    Xa_cycle_initial_history.append(np.asarray(Xa).copy())

    # Long forecast: classify each member and reuse step Ta as the short prior.
    xf_4d_box = np.zeros((das.edim, das.xdim, position + 1))
    Xf = np.matrix(np.zeros((das.xdim, das.edim)))
    unchange_ens = []
    change_ens = []
    for k in range(das.edim):
        traj = run_RK4(
            lorenz63,
            10.0,
            8.0 / 3.0,
            28.0,
            np.ravel(Xa[:, k]),
            fcst_dt,
            fcst_dt * position,
        ).integrate()
        xf_4d_box[k] = traj[:position + 1, :].T

        if np.all(xf_4d_box[k, 0, :] > 0) or np.all(xf_4d_box[k, 0, :] < 0):
            unchange_ens.append(k)
        else:
            change_ens.append(k)

        Xf[:, k] = np.matrix(xf_4d_box[k, :, Ta]).T

    p_change = len(change_ens)
    p_change_history.append(p_change)
    xf_4d_box_history.append(xf_4d_box.copy())
    unchange_ens_history.append(unchange_ens.copy())
    change_ens_history.append(change_ens.copy())

    if p_change == 0:
        branch = 'free'
        branch_counts[branch] += 1
        nature_traj = run_RK4(
            lorenz63,
            10.0,
            8.0 / 3.0,
            28.0,
            x_original,
            fcst_dt,
            fcst_dt * Ta,
        ).integrate()
        evo_history[:, time * Ta + 1:(time + 1) * Ta + 1] = nature_traj[1:Ta + 1, :].T
        x_original = nature_traj[Ta, :].copy()
        past_safe_metadata.append(None)
        perturbation_norm_history.append(None)

    elif p_change < das.edim:
        branch = 'current_safe_control'
        branch_counts[branch] += 1
        cycle_perturbed = True
        safe_member = int(rng.choice(unchange_ens))
        shift_member = int(rng.choice(change_ens))
        x_final, perturbation_history, perturbation_vector = build_and_apply_perturbation(
            x_original,
            xf_4d_box[safe_member],
            xf_4d_box[shift_member],
            D,
            Ta,
            solver_params,
            control_indices=CONTROL_INDICES,
        )
        evo_history[:, time * Ta:time * Ta + Ta] = perturbation_history.T
        evo_history[:, (time + 1) * Ta] = x_final
        perturbed_time.extend(range(time * Ta + 1, time * Ta + Ta))
        x_original = x_final.copy()
        past_safe_metadata.append(None)
        perturbation_norm_history.append(np.linalg.norm(perturbation_vector, axis=0))

    else:
        branch = 'past_safe_control'
        branch_counts[branch] += 1
        cycle_perturbed = True
        regime_sign = np.sign(x_original[0])
        xf_safe_traj, safe_s, safe_phase, safe_member, aligned_margin, full_margin, safe_pool_size = find_past_safe_trajectory(
            time=time,
            Xa_history_control=Xa_history_control,
            osse_Xa_history=osse_Xa_history,
            osse_start_index=osse_start_index,
            position=position,
            Ta=Ta,
            regime_sign=regime_sign,
            solver_params=solver_params,
            max_depth=400,
            path_figures=PathFigures,
            x_nature=x_nature,
            t0=t0,
            candidate_pool='full',
        )
        shift_member = int(rng.choice(change_ens))
        plot_found_past_safe_trajectory(
            time=time,
            xf_4d_box=xf_4d_box,
            xf_safe_traj=xf_safe_traj,
            safe_s=safe_s,
            safe_phase=safe_phase,
            safe_member=safe_member,
            aligned_margin=aligned_margin,
            full_margin=full_margin,
            t0=t0,
            Ta=Ta,
            position=position,
            x_nature=x_nature,
            t_start=t_start,
            path_figures=PathFigures,
            show=False,
            save_figs=SAVE_CYCLE_FIGS,
        )
        x_final, perturbation_history, perturbation_vector = build_and_apply_perturbation(
            x_original,
            xf_safe_traj,
            xf_4d_box[shift_member],
            D,
            Ta,
            solver_params,
            control_indices=CONTROL_INDICES,
        )
        evo_history[:, time * Ta:time * Ta + Ta] = perturbation_history.T
        evo_history[:, (time + 1) * Ta] = x_final
        perturbed_time.extend(range(time * Ta + 1, time * Ta + Ta))
        x_original = x_final.copy()
        safe_search_log.append({
            'time': time,
            's': safe_s,
            'phase': safe_phase,
            'member': safe_member,
            'aligned_margin': aligned_margin,
            'full_margin': full_margin,
            'candidate_pool': 'full',
            'pool_size': safe_pool_size,
        })
        past_safe_metadata.append({
            's': safe_s,
            'phase': safe_phase,
            'safe_member': safe_member,
            'shift_member': shift_member,
            'aligned_margin': aligned_margin,
            'full_margin': full_margin,
            'candidate_pool': 'full',
            'pool_size': safe_pool_size,
        })
        perturbation_norm_history.append(np.linalg.norm(perturbation_vector, axis=0))

    if cycle_perturbed:
        cycle_perturbed_steps = range(time * Ta + 1, time * Ta + Ta)
        if np.all(evol[0, :] * initial_regime_value > 0):
            false_alarm_cycles.append(time)
            false_alarm_perturbed_time.extend(cycle_perturbed_steps)
        else:
            success_candidate_cycles.append(time)
            success_perturbed_time.extend(cycle_perturbed_steps)

    hidden_forecast_needs_recompute = cycle_perturbed
    branch_history.append(branch)

    # Assimilation uses the controlled truth plus new synthetic observation noise.
    yo = x_original + rng.normal(0, obs_error_std, das.xdim)
    yo_history.append(yo.copy())
    Xa, KH = das.compute_analysis(Xf, yo)
    Xa_history_control[time + 1] = Xa

    xa_mean = np.asarray(Xa).mean(axis=1)
    analysis_time_index = (time + 1) * Ta
    controlled_truth_at_analysis = evo_history[:, analysis_time_index]
    free_truth_at_analysis = x_nature[:, t0 + analysis_time_index]
    analysis_rmse.append(
        np.linalg.norm(xa_mean - controlled_truth_at_analysis) / np.sqrt(das.xdim)
    )
    counterfactual_divergence.append(
        np.linalg.norm(xa_mean - free_truth_at_analysis) / np.sqrt(das.xdim)
    )

analysis_rmse = np.asarray(analysis_rmse)
counterfactual_divergence = np.asarray(counterfactual_divergence)
p_change_history = np.asarray(p_change_history)
N_COMPLETED = len(branch_history)
p_change_distribution = {p: int(np.sum(p_change_history == p)) for p in range(das.edim + 1)}
shift_cycles = np.where(p_change_history > 0)[0]
first_shift_cycle = int(shift_cycles[0]) if shift_cycles.size else None
perturbed_time = np.asarray(perturbed_time, dtype=int)
false_alarm_perturbed_time = np.asarray(false_alarm_perturbed_time, dtype=int)
success_perturbed_time = np.asarray(success_perturbed_time, dtype=int)
control_cycle_count = int(np.sum(np.asarray(branch_history) != 'free'))
control_ratio = control_cycle_count / N_COMPLETED if N_COMPLETED else np.nan
total_perturbed_steps = int(perturbed_time.size)
total_false_alarm_steps = int(false_alarm_perturbed_time.size)
total_success_justified_steps = int(success_perturbed_time.size)
total_completed_steps = int(N_COMPLETED * Ta)
success = not cant_control
false_alarm_ratio = (
    total_false_alarm_steps / total_perturbed_steps
    if total_perturbed_steps else np.nan
)
success_control_ratio = (
    total_success_justified_steps / total_completed_steps
    if total_completed_steps else np.nan
)

print('success:', success)
print('cant_control:', cant_control)
print('Cycle until failure:', cant_control_cycle if cant_control else None)
print('False alarm cycles:', len(false_alarm_cycles))
print('Control ratio:', control_ratio)
print('False alarm ratio:', false_alarm_ratio)
print('Success control ratio:', success_control_ratio)
print('Perturbed steps:', total_perturbed_steps)
print('False alarm perturbed steps:', total_false_alarm_steps)
print('Success-justified perturbed steps:', total_success_justified_steps)
print('Branch counts:', branch_counts)
print('p_change distribution:', p_change_distribution)
print('First cycle with p_change > 0:', first_shift_cycle)
print('Safe search log:')
print('time | s_found | phase | member | aligned_margin | full_margin')
for row in safe_search_log:
    print(
        f"{row['time']:4d} | {row['s']:7d} | {row['phase']:>5s} | {row['member']:6d} | "
        f"{row['aligned_margin']:14.6f} | {row['full_margin']:11.6f}"
    )
if N_COMPLETED:
    print('Mean analysis RMSE vs NR controlled:', float(np.mean(analysis_rmse)))
    print('Analysis RMSE vs NR controlled min/max:', float(np.min(analysis_rmse)), float(np.max(analysis_rmse)))
    print('Mean analysis-vs-NR separation:', float(np.mean(counterfactual_divergence)))
    print(
        'Analysis-vs-NR separation min/max:',
        float(np.min(counterfactual_divergence)),
        float(np.max(counterfactual_divergence)),
    )


## Plot Controlled NR vs Free NR


In [ ]:
plot_steps = np.arange(N_COMPLETED * Ta + 1)
cycle_steps = np.arange(N_COMPLETED) * Ta
obs_steps = np.arange(1, N_COMPLETED + 1) * Ta

controlled_traj = evo_history[:, :N_COMPLETED * Ta + 1]
free_traj = x_nature[:, t0:t0 + N_COMPLETED * Ta + 1]
cycle_controlled = evo_history[:, cycle_steps]

y_obs_array = np.asarray(y_obs)
if y_obs_array.shape[0] == das.xdim:
    osse_obs = y_obs_array[:, t0 + obs_steps]
else:
    osse_obs = y_obs_array[t0 + obs_steps, :].T
cse_obs = np.asarray(yo_history).T

branch_history_array = np.asarray(branch_history)
free_mask = branch_history_array == 'free'
current_safe_mask = branch_history_array == 'current_safe_control'
past_safe_mask = branch_history_array == 'past_safe_control'

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True, constrained_layout=True)
var_labels = ['x', 'y', 'z']

for j, ax in enumerate(axes):
    ax.plot(
        plot_steps,
        controlled_traj[j],
        color='red',
        linewidth=2.0,
        label='NR controlled' if j == 0 else None,
    )
    ax.plot(
        plot_steps,
        free_traj[j],
        color='black',
        linewidth=1.8,
        label='NR' if j == 0 else None,
    )
    ax.axhline(0, color='gray', linestyle='--', linewidth=1.0)

    if np.any(free_mask):
        ax.scatter(
            cycle_steps[free_mask],
            cycle_controlled[j, free_mask],
            color='tab:blue',
            s=45,
            zorder=5,
            label='no control required' if j == 0 else None,
        )
    if np.any(current_safe_mask):
        ax.scatter(
            cycle_steps[current_safe_mask],
            cycle_controlled[j, current_safe_mask],
            color='green',
            marker='o',
            s=50,
            zorder=6,
            label='control applied' if j == 0 else None,
        )
    if np.any(past_safe_mask):
        ax.scatter(
            cycle_steps[past_safe_mask],
            cycle_controlled[j, past_safe_mask],
            color='green',
            marker='D',
            edgecolor='black',
            linewidth=0.6,
            s=55,
            zorder=7,
            label='control applied: all members shift' if j == 0 else None,
        )

    ax.scatter(
        obs_steps,
        osse_obs[j],
        facecolors='none',
        edgecolors='black',
        marker='o',
        s=30,
        linewidth=0.9,
        zorder=4,
        label='OSSE observations around NR' if j == 0 else None,
    )
    ax.scatter(
        obs_steps,
        cse_obs[j],
        facecolors='none',
        edgecolors='tab:purple',
        marker='s',
        s=30,
        linewidth=0.9,
        zorder=8,
        label='CSE observations around NR controlled' if j == 0 else None,
    )

    if perturbed_time.size:
        marker_times = perturbed_time[perturbed_time < N_COMPLETED * Ta]
        ax.scatter(
            marker_times,
            np.full(marker_times.shape, 0.025),
            color='gray',
            s=12,
            alpha=0.75,
            transform=ax.get_xaxis_transform(),
            clip_on=False,
            label='perturbed steps' if j == 0 else None,
        )

    ax.set_ylabel(var_labels[j])
    ax.grid(True, alpha=0.18)

axes[-1].set_xlabel('Relative time step from t0')
axes[0].set_title('NR controlled vs NR')
axes[0].legend(loc='best', ncols=2)
fig.savefig(PathFigures / 'nr_controlled_vs_nr_qiwenevents.png', dpi=200, bbox_inches='tight')
plt.show()


## 3D Plot Controlled NR vs Free NR


In [ ]:
# 3D Lorenz-attractor view of the controlled-vs-free result.
# This cell can be rerun without rerunning the CSE loop: if evo_history is not
# available in memory, it loads the selected event from the saved 40-event pickle.
from pathlib import Path
import importlib
import pickle

import numpy as np

from class_da_sys import da_system
import class_cse_qiwen as cse_qiwen

# Reload the helper module so this cell can use functions added after the
# current kernel first imported class_cse_qiwen.
cse_qiwen = importlib.reload(cse_qiwen)
plot_3d_nature_run_comparison = cse_qiwen.plot_3d_nature_run_comparison

BASE_PATH = globals().get('BASE_PATH', Path.cwd())
PathFigures = globals().get('PathFigures', BASE_PATH / 'CSE_Qiwen_xyz_control_nature_qiwenevents')
PathFigures.mkdir(parents=True, exist_ok=True)

if 'x_nature' not in globals() or 'Ta' not in globals():
    das_plot = da_system().load(BASE_PATH / 'x_analysis_init.pkl')
    x_nature = das_plot.getStateVector().getTrajectory()
    Ta = das_plot.acyc_step

if 'evo_history' in globals() and 'N_COMPLETED' in globals() and 't0' in globals():
    plot_evo_history = evo_history
    plot_t0 = t0
    plot_num_cycles = N_COMPLETED
else:
    plot_D = globals().get('SUMMARY_D', globals().get('D', 0.05))
    plot_event_index = globals().get('EVENT_INDEX', 0)
    plot_results_file = BASE_PATH / f'cse_qiwen_xyz_control_40_results_D{plot_D}{OUTPUT_TAG}.pkl'
    with open(plot_results_file, 'rb') as f:
        plot_results = pickle.load(f)
    plot_result = next(row for row in plot_results if row['event_index'] == plot_event_index)
    plot_evo_history = np.asarray(plot_result['evo_history'])
    plot_t0 = plot_result['t0']
    plot_num_cycles = min(
        globals().get('experiment_step', (plot_evo_history.shape[1] - 1) // Ta),
        (plot_evo_history.shape[1] - 1) // Ta,
    )
    print(
        'Loaded controlled trajectory from:', plot_results_file,
        '| event_index:', plot_event_index,
        '| num_cycles:', plot_num_cycles,
    )

x_nature_nocontrol_3d = x_nature[:, plot_t0:plot_t0 + plot_num_cycles * Ta + 1]
x_nature_control_3d = plot_evo_history[:, :plot_num_cycles * Ta + 1]

fig_3d_nr, axes_3d_nr, nr_3d_path = plot_3d_nature_run_comparison(
    nature_nocontrol=x_nature_nocontrol_3d,
    nature_control=x_nature_control_3d,
    path_figures=PathFigures,
    Ta=Ta,
    num_cycles=plot_num_cycles,
    output_filename='NR_3D_comparison_after_control_qiwenevents.png',
)


## Filter Diagnostic During Control


In [ ]:
# Post-run filter diagnostic.
# This uses histories produced by the Controlled CSE Loop to check how the
# analysis behaved during free vs controlled cycles: tracking of x, innovation
# versus prior spread, and analysis RMSE by branch.
fig_filter_diagnostic, axes_filter_diagnostic, filter_diagnostic = plot_filter_diagnostic_during_control(
    evo_history=evo_history,
    Xa_history_control=Xa_history_control,
    xf_4d_box_history=xf_4d_box_history,
    yo_history=yo_history,
    branch_history=branch_history,
    analysis_rmse=analysis_rmse,
    N_COMPLETED=N_COMPLETED,
    Ta=Ta,
    xdim=das.xdim,
    path_figures=PathFigures,
    output_filename='filter_diagnostic_during_control_qiwenevents.png',
)


## Past Safe Trajectory Test


In [ ]:
# Unit test and visual diagnostic for the past-safe fallback.
# It checks that, when all cycle-0 members predict a shift, the search finds
# a valid OSSE safe trajectory and plots its true origin relative to the
# current CSE cycle. The exact s/member can change if the upstream loop state
# is rerun with different control settings, so this diagnostic does not pin
# those values.

# Reload the module to pick up the updated test function
import importlib
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)
run_past_safe_trajectory_test = cse_qiwen.run_past_safe_trajectory_test

past_safe_trajectory_test = run_past_safe_trajectory_test(
    cycle=0,
    x_nature=x_nature,
    t0=t0,
    t_start=t_start,
    Xa_history_control=Xa_history_control,
    osse_Xa_history=osse_Xa_history,
    osse_start_index=osse_start_index,
    osse_summary=osse_summary,
    xf_4d_box_history=xf_4d_box_history,
    position=position,
    Ta=Ta,
    xdim=das.xdim,
    solver_params=solver_params,
    path_figures=PathFigures,
    expected_phase='osse',
    expected_s=None,
    expected_member=None,
    output_filename='past_safe_trajectory_test_cycle_000_qiwenevents.png',
)

## Visual Diagnostic of the Past Safe Search


In [ ]:
# Visual diagnostic for the past-safe search.
# For s = 1..20 cycles back, this re-integrates candidate analysis members,
# aligns them with the current CSE horizon, and counts which members stay in
# the desired regime.
past_safe_search_diagnostic = plot_past_safe_search_visual_diagnostic(
    time=0,
    x_nature=x_nature,
    t0=t0,
    Xa_history_control=Xa_history_control,
    osse_Xa_history=osse_Xa_history,
    osse_start_index=osse_start_index,
    osse_summary=osse_summary,
    position=position,
    Ta=Ta,
    solver_params=solver_params,
    path_figures=PathFigures,
    max_depth=20,
    output_filename='past_safe_search_visual_diagnostic_qiwenevents.png',
)


## Per-Cycle Visual Diagnostics


In [ ]:
# Per-cycle diagnostic plots for selected CSE cycles.
# Each figure shows: (1) nature + observation + initial ensemble,
# (2) the full free forecast ensemble, and (3) the Qiwen classification
# into changed vs unchanged members.
CYCLES_TO_PLOT = range(2)  # Change to range(N_COMPLETED) to save/show all completed cycles.
cse_cycle_diagnostic_figures = plot_cse_cycle_diagnostics(
    cycles_to_plot=CYCLES_TO_PLOT,
    cycle_t_now_history=cycle_t_now_history,
    Xa_cycle_initial_history=Xa_cycle_initial_history,
    xf_4d_box_history=xf_4d_box_history,
    yo_history=yo_history,
    change_ens_history=change_ens_history,
    unchange_ens_history=unchange_ens_history,
    x_nature=x_nature,
    t0=t0,
    t_start=t_start,
    Ta=Ta,
    position=position,
    edim=das.edim,
    path_figures=PathFigures,
    show=True,
)
for cycle in CYCLES_TO_PLOT:
    cycle_path = PathFigures / f'cse_cycle_{cycle:03d}_diagnostic.png'
    cycle_path.replace(PathFigures / f'cse_cycle_{cycle:03d}_diagnostic_qiwenevents.png')


## Batch Helper: Single CSE Event


In [ ]:
# Batch helper for one complete xyz-control CSE event.
# The implementation lives in class_cse_qiwen.py; this wrapper binds the
# notebook's loaded OSSE/nature-run objects so later cells can keep calling
# run_single_cse(event_index, D_run, seed, regime_reference='nature', candidate_pool='full').
def run_single_cse(event_index, D_run, seed, regime_reference='nature', candidate_pool='full'):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        control_indices=CONTROL_INDICES,
    )


## Verification: Regime Reference Mode


In [ ]:
# Small verification for run_single_cse search-reference options.
# This does one event only; it does not launch the full batch.
from pathlib import Path
import importlib
import pickle

import numpy as np

import class_cse_qiwen as cse_qiwen

cse_qiwen = importlib.reload(cse_qiwen)
run_single_cse_core = cse_qiwen.run_single_cse

VERIFY_EVENT_INDEX = globals().get('VERIFY_EVENT_INDEX', 0)
VERIFY_D_RUN = globals().get('VERIFY_D_RUN', 0.3)
VERIFY_SEED = globals().get('VERIFY_SEED', 1000)


def run_single_cse(
    event_index,
    D_run,
    seed,
    regime_reference='nature',
    candidate_pool='full',
):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        control_indices=CONTROL_INDICES,
    )

r_full = run_single_cse(event_index=VERIFY_EVENT_INDEX, D_run=VERIFY_D_RUN, seed=VERIFY_SEED)

stored_file = BASE_PATH / f'cse_qiwen_xyz_control_40_results_D{VERIFY_D_RUN}{OUTPUT_TAG}.pkl'
stored_entry = None
if stored_file.exists():
    with open(stored_file, 'rb') as f:
        stored_results = pickle.load(f)
    stored_matches = [row for row in stored_results if row['event_index'] == VERIFY_EVENT_INDEX]
    if stored_matches:
        stored_entry = stored_matches[0]

if stored_entry is None:
    print(f'no stored reference for D={VERIFY_D_RUN}, skipping regression check')
else:
    scalar_checks = {
        'success': r_full['success'] == stored_entry['success'],
        'control_ratio': np.array_equal(np.asarray(r_full['control_ratio']), np.asarray(stored_entry['control_ratio'])),
        'false_alarm_ratio': np.array_equal(np.asarray(r_full['false_alarm_ratio']), np.asarray(stored_entry['false_alarm_ratio'])),
    }
    if 'analysis_rmse' in stored_entry:
        scalar_checks['analysis_rmse'] = np.array_equal(r_full['analysis_rmse'], stored_entry['analysis_rmse'])
    else:
        print(f'Stored D={VERIFY_D_RUN} event {VERIFY_EVENT_INDEX} is a legacy result without analysis_rmse array.')
        print('Checking mean_analysis_rmse and max_analysis_rmse instead.')
        scalar_checks['mean_analysis_rmse'] = np.array_equal(
            np.asarray(r_full['mean_analysis_rmse']),
            np.asarray(stored_entry['mean_analysis_rmse']),
        )
        scalar_checks['max_analysis_rmse'] = np.array_equal(
            np.asarray(r_full['max_analysis_rmse']),
            np.asarray(stored_entry['max_analysis_rmse']),
        )

    print(f'Default nature/full-mode comparison with stored D={VERIFY_D_RUN} event {VERIFY_EVENT_INDEX}:')
    for key, value in scalar_checks.items():
        print(f'  {key}: {value}')
    assert all(scalar_checks.values()), 'Default full-pool run does not reproduce the stored metrics.'

r_unchanged_bkp = run_single_cse(
    event_index=VERIFY_EVENT_INDEX,
    D_run=VERIFY_D_RUN,
    seed=VERIFY_SEED,
    candidate_pool='unchanged_bkp',
)


def _compact_safe_log(rows):
    return [
        {
            'time': row['time'],
            's': row['s'],
            'phase': row['phase'],
            'member': row['member'],
            'pool_size': row.get('pool_size'),
            'regime_reference': row.get('regime_reference'),
            'regime_sign': row.get('regime_sign'),
            'candidate_pool': row.get('candidate_pool'),
        }
        for row in rows
    ]

print('\nPast-safe search logs side by side:')
print('full         :', _compact_safe_log(r_full['safe_search_log']))
print('unchanged_bkp:', _compact_safe_log(r_unchanged_bkp['safe_search_log']))


## Batch Run: 40 CSE Events


In [ ]:
# Run the xyz-control CSE validation over the first 40 selected events.
# This batch runs only the missing D value needed for the Figure 3(a)-style
# comparison. The pickle for each D is updated after each event, so interrupted
# runs can resume safely.
import pickle
import time

D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
REGIME_REFERENCE = 'nature'
#CANDIDATE_POOL = 'unchanged_bkp'
CANDIDATE_POOL = 'full'
N_BATCH_EVENTS = 40

for D_run in D_VALUES:
    output_suffix = '' if (REGIME_REFERENCE == 'nature' and CANDIDATE_POOL == 'full') else f'_{REGIME_REFERENCE}_{CANDIDATE_POOL}'
    output_file = BASE_PATH / f'cse_qiwen_xyz_control_40_results_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'
    if output_file.exists():
        with open(output_file, 'rb') as f:
            results = pickle.load(f)
        print(f'Resuming {output_file}: {len(results)}/{N_BATCH_EVENTS} events already done')
    else:
        results = []

    for i in range(len(results), N_BATCH_EVENTS):
        t_event_start = time.time()
        results.append(run_single_cse(
            i,
            D_run,
            seed=1000 + i,
            regime_reference=REGIME_REFERENCE,
            candidate_pool=CANDIDATE_POOL,
        ))
        with open(output_file, 'wb') as f:
            pickle.dump(results, f)
        elapsed_min = (time.time() - t_event_start) / 60.0
        print(f'Saved partial result {i + 1}/{N_BATCH_EVENTS} for D={D_run} ({elapsed_min:.2f} min)')

    print('Saved:', output_file)


## Batch Run: Figure 3(a) T-D Grid


In [ ]:
# Run the xyz-control validation over the full Figure 3(a) grid: rows T/T0 and columns D.
# This is resumable: every (T, D) pair is stored in its own pickle and updated after each event.
import pickle
import time

FIG3A_T_MULTIPLES = {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
}
FIG3A_D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
FIG3A_N_BATCH_EVENTS = 40
FIG3A_REGIME_REFERENCE = 'nature'
FIG3A_CANDIDATE_POOL = 'full'


def _fig3a_t_file_label(T_label):
    return (
        T_label.strip('[]')
        .replace('.', 'p')
        .replace('T0', 'T0')
    )


def _fig3a_output_file(T_label, D_run):
    T_file_label = _fig3a_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3A_REGIME_REFERENCE == 'nature' and FIG3A_CANDIDATE_POOL == 'full')
        else f'_{FIG3A_REGIME_REFERENCE}_{FIG3A_CANDIDATE_POOL}'
    )
    return BASE_PATH / f'cse_qiwen_xyz_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def run_single_cse_fig3a(event_index, D_run, seed, position_run, T_label, multiple_run):
    row = run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position_run,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=FIG3A_N_BATCH_EVENTS,
        regime_reference=FIG3A_REGIME_REFERENCE,
        candidate_pool=FIG3A_CANDIDATE_POOL,
        control_indices=CONTROL_INDICES,
    )
    row['T_label'] = T_label
    row['multiple_number'] = float(multiple_run)
    row['position'] = int(position_run)
    return row


for T_label, multiple_run in FIG3A_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    print(f'=== T={T_label}, multiple={multiple_run}, position={position_run} ===')
    for D_run in FIG3A_D_VALUES:
        output_file = _fig3a_output_file(T_label, D_run)
        if output_file.exists():
            with open(output_file, 'rb') as f:
                results = pickle.load(f)
            print(f'Resuming {output_file.name}: {len(results)}/{FIG3A_N_BATCH_EVENTS} events already done')
        else:
            results = []

        for i in range(len(results), FIG3A_N_BATCH_EVENTS):
            t_event_start = time.time()
            results.append(run_single_cse_fig3a(
                event_index=i,
                D_run=D_run,
                seed=1000 + i,
                position_run=position_run,
                T_label=T_label,
                multiple_run=multiple_run,
            ))
            with open(output_file, 'wb') as f:
                pickle.dump(results, f)
            elapsed_min = (time.time() - t_event_start) / 60.0
            print(
                f'Saved partial result {i + 1}/{FIG3A_N_BATCH_EVENTS} '
                f'for T={T_label}, D={D_run} ({elapsed_min:.2f} min)'
            )

        print('Saved:', output_file)


## Figure 3(a) Grid Summary and Comparison


In [ ]:
# Summarize the full Figure 3(a) T-D grid and compare with the Qiwen control-xyz table.
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = globals().get('BASE_PATH', Path.cwd())
PathFigures = globals().get('PathFigures', BASE_PATH / 'CSE_Qiwen_xyz_control_nature_qiwenevents')
PathFigures.mkdir(parents=True, exist_ok=True)
T_0 = globals().get('T_0', 75.1)
OUTPUT_TAG = globals().get('OUTPUT_TAG', '_qiwenevents')

FIG3A_T_MULTIPLES = globals().get('FIG3A_T_MULTIPLES', {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
})
FIG3A_D_VALUES = globals().get('FIG3A_D_VALUES', [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
FIG3A_N_BATCH_EVENTS = globals().get('FIG3A_N_BATCH_EVENTS', 40)
FIG3A_REGIME_REFERENCE = globals().get('FIG3A_REGIME_REFERENCE', 'nature')
FIG3A_CANDIDATE_POOL = globals().get('FIG3A_CANDIDATE_POOL', 'full')

# Values transcribed from Figure 3(a) of the paper (control x,y,z), T×D grid.
QIWEN_FIG3A_CONTROL_XYZ_VALUES = globals().get('QIWEN_FIG3A_CONTROL_XYZ_VALUES', {
    '[1.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.0, 0.3: 0.0, 0.4: 0.0, 0.5: 0.025},
    '[2T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.025, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[2.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.025, 0.1: 0.250, 0.2: 0.275, 0.3: 0.250, 0.4: 0.200, 0.5: 0.175},
    '[3T0]': {0.02: 0.0, 0.03: 0.050, 0.04: 0.425, 0.05: 0.550, 0.1: 0.800, 0.2: 0.675, 0.3: 0.400, 0.4: 0.100, 0.5: 0.200},
    '[4T0]': {0.02: 0.050, 0.03: 0.975, 0.04: 0.925, 0.05: 0.975, 0.1: 0.825, 0.2: 0.825, 0.3: 0.725, 0.4: 0.500, 0.5: 0.525},
})


def _fig3a_t_file_label(T_label):
    return T_label.strip('[]').replace('.', 'p').replace('T0', 'T0')


def _fig3a_output_file(T_label, D_run):
    T_file_label = _fig3a_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3A_REGIME_REFERENCE == 'nature' and FIG3A_CANDIDATE_POOL == 'full')
        else f'_{FIG3A_REGIME_REFERENCE}_{FIG3A_CANDIDATE_POOL}'
    )
    return BASE_PATH / f'cse_qiwen_xyz_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def _legacy_4T0_output_file(D_run):
    return BASE_PATH / f'cse_qiwen_xyz_control_40_results_D{D_run}{OUTPUT_TAG}.pkl'


event_rows = []
aggregate_rows = []
for T_label, multiple_run in FIG3A_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    for D_run in FIG3A_D_VALUES:
        result_file = _fig3a_output_file(T_label, D_run)
        if not result_file.exists() and T_label == '[4T0]':
            legacy_file = _legacy_4T0_output_file(D_run)
            if legacy_file.exists():
                result_file = legacy_file
        if not result_file.exists():
            print('Missing Figure 3(a) grid result file:', result_file)
            aggregate_rows.append({
                'T': T_label,
                'multiple_number': multiple_run,
                'position': position_run,
                'D': D_run,
                'n_events': 0,
                'success_ratio': np.nan,
                'mean_control_ratio': np.nan,
                'mean_false_alarm_ratio': np.nan,
                'mean_success_control_ratio': np.nan,
                'n_failures': np.nan,
                'result_file': str(result_file),
            })
            continue

        with open(result_file, 'rb') as f:
            rows = pickle.load(f)
        for row in rows:
            event_rows.append({
                'T': row.get('T_label', T_label),
                'multiple_number': row.get('multiple_number', multiple_run),
                'position': row.get('position', position_run),
                'D': D_run,
                'event': row['event_index'],
                'success': row['success'],
                'control_ratio': row['control_ratio'],
                'false_alarm_ratio': row['false_alarm_ratio'],
                'success_control_ratio': row['success_control_ratio'],
                'failure_cycle': row['failure_cycle'],
                'result_file': str(result_file),
            })
        aggregate_rows.append({
            'T': T_label,
            'multiple_number': multiple_run,
            'position': position_run,
            'D': D_run,
            'n_events': len(rows),
            'success_ratio': float(np.mean([row['success'] for row in rows])) if rows else np.nan,
            'mean_control_ratio': float(np.nanmean([row['control_ratio'] for row in rows])) if rows else np.nan,
            'mean_false_alarm_ratio': float(np.nanmean([row['false_alarm_ratio'] for row in rows])) if rows else np.nan,
            'mean_success_control_ratio': float(np.nanmean([row['success_control_ratio'] for row in rows])) if rows else np.nan,
            'n_failures': int(np.sum([not row['success'] for row in rows])) if rows else np.nan,
            'result_file': str(result_file),
        })

fig3a_event_table = pd.DataFrame(event_rows)
fig3a_aggregate_table = pd.DataFrame(aggregate_rows)
fig3a_success_matrix = fig3a_aggregate_table.pivot(index='T', columns='D', values='success_ratio')
fig3a_success_matrix = fig3a_success_matrix.reindex(index=list(FIG3A_T_MULTIPLES), columns=FIG3A_D_VALUES)

qiwen_fig3a_matrix = pd.DataFrame.from_dict(QIWEN_FIG3A_CONTROL_XYZ_VALUES, orient='index')
qiwen_fig3a_matrix = qiwen_fig3a_matrix.reindex(index=list(FIG3A_T_MULTIPLES), columns=FIG3A_D_VALUES)
fig3a_difference_matrix = fig3a_success_matrix - qiwen_fig3a_matrix

fig3a_event_table_file = BASE_PATH / 'cse_qiwen_xyz_control_40_fig3a_grid_events_nature_qiwenevents.csv'
fig3a_aggregate_table_file = BASE_PATH / 'cse_qiwen_xyz_control_40_fig3a_grid_aggregates_nature_qiwenevents.csv'
fig3a_success_matrix_file = BASE_PATH / 'cse_qiwen_xyz_control_40_fig3a_success_matrix_nature_qiwenevents.csv'
fig3a_difference_matrix_file = BASE_PATH / 'cse_qiwen_xyz_control_40_fig3a_minus_qiwen_matrix_nature_qiwenevents.csv'
fig3a_event_table.to_csv(fig3a_event_table_file, index=False)
fig3a_aggregate_table.to_csv(fig3a_aggregate_table_file, index=False)
fig3a_success_matrix.to_csv(fig3a_success_matrix_file)
fig3a_difference_matrix.to_csv(fig3a_difference_matrix_file)

try:
    display(fig3a_success_matrix)
    display(fig3a_difference_matrix)
except NameError:
    print('This-run success matrix:')
    print(fig3a_success_matrix)
    print('This-run minus Qiwen matrix:')
    print(fig3a_difference_matrix)

PAPER_TABLE_COLORS = {
    'ge_08': '#9c7a00',
    'ge_06': '#d1a000',
    'ge_04': '#ffe173',
    'ge_02': '#fff0b3',
    'gt_00': '#fff8dc',
    'zero': '#ffffff',
    'missing': '#f2f2f2',
}


def _paper_cell_color(value):
    if pd.isna(value):
        return PAPER_TABLE_COLORS['missing']
    value = float(value)
    if np.isclose(value, 0.0):
        return PAPER_TABLE_COLORS['zero']
    if value >= 0.8:
        return PAPER_TABLE_COLORS['ge_08']
    if value >= 0.6:
        return PAPER_TABLE_COLORS['ge_06']
    if value >= 0.4:
        return PAPER_TABLE_COLORS['ge_04']
    if value >= 0.2:
        return PAPER_TABLE_COLORS['ge_02']
    return PAPER_TABLE_COLORS['gt_00']


def _format_paper_value(value):
    if pd.isna(value):
        return ''
    value = float(value)
    if np.isclose(value, 0.0):
        return '0'
    return f'{value:.3f}'.rstrip('0').rstrip('.')


def _format_D_label(value):
    return f'{float(value):.2f}'.rstrip('0').rstrip('.')


def _draw_paper_table(ax, matrix, title):
    ax.axis('off')
    header = [r'T \\ D'] + [_format_D_label(D_run) for D_run in matrix.columns]
    table_text = [header]
    table_colors = [['#ffffff'] * len(header)]

    for T_label in matrix.index:
        row_text = [T_label]
        row_colors = ['#ffffff']
        for D_run in matrix.columns:
            value = matrix.loc[T_label, D_run]
            row_text.append(_format_paper_value(value))
            row_colors.append(_paper_cell_color(value))
        table_text.append(row_text)
        table_colors.append(row_colors)

    mpl_table = ax.table(
        cellText=table_text,
        cellColours=table_colors,
        cellLoc='center',
        loc='center',
        bbox=[0.0, 0.0, 1.0, 0.88],
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8.4)
    mpl_table.scale(1.0, 1.16)

    for (row, col), table_cell in mpl_table.get_celld().items():
        table_cell.set_edgecolor('black')
        table_cell.set_linewidth(0.65)
        if row == 0 or col == 0:
            table_cell.set_text_props(weight='normal', color='black')
            table_cell.set_facecolor('#ffffff')
        elif table_cell.get_facecolor()[:3] == (0.611764705882353, 0.47843137254901963, 0.0):
            table_cell.set_text_props(color='white')

    ax.text(0.5, 0.945, title, ha='center', va='center', fontsize=10)
    return mpl_table


fig = plt.figure(figsize=(11.8, 4.5), constrained_layout=False)
grid = fig.add_gridspec(2, 2, height_ratios=[0.22, 1.0], hspace=0.12, wspace=0.08)
legend_ax = fig.add_subplot(grid[0, :])
legend_ax.axis('off')
legend_labels = [[r'$\geq 0.8$', r'$\geq 0.6$', r'$\geq 0.4$', r'$\geq 0.2$', r'$> 0$', r'$0$']]
legend_colors = [[
    PAPER_TABLE_COLORS['ge_08'],
    PAPER_TABLE_COLORS['ge_06'],
    PAPER_TABLE_COLORS['ge_04'],
    PAPER_TABLE_COLORS['ge_02'],
    PAPER_TABLE_COLORS['gt_00'],
    PAPER_TABLE_COLORS['zero'],
]]
legend_table = legend_ax.table(
    cellText=legend_labels,
    cellColours=legend_colors,
    cellLoc='center',
    loc='center',
    bbox=[0.38, 0.18, 0.24, 0.62],
)
legend_table.auto_set_font_size(False)
legend_table.set_fontsize(8.2)
for (row, col), table_cell in legend_table.get_celld().items():
    table_cell.set_edgecolor('black')
    table_cell.set_linewidth(0.65)
    if col == 0:
        table_cell.set_text_props(color='white')

axes = [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])]
_draw_paper_table(axes[0], qiwen_fig3a_matrix, '(a) Qiwen control xyz')
_draw_paper_table(axes[1], fig3a_success_matrix, '(a) Diego control xyz')

fig3a_grid_png = PathFigures / 'cse_qiwen_xyz_control_fig3a_grid_comparison_nature_qiwenevents.png'
fig.savefig(fig3a_grid_png, dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved event table:', fig3a_event_table_file)
print('Saved aggregate table:', fig3a_aggregate_table_file)
print('Saved success matrix:', fig3a_success_matrix_file)
print('Saved difference matrix:', fig3a_difference_matrix_file)
print('Saved comparison figure:', fig3a_grid_png)
